# Intersection

PPO baseline for HighwayEnv intersection: scalar acceleration control, native speed cap, deterministic route-following steering.

## 1. Setup

### 1.1 Imports

In [ ]:
from __future__ import annotations

import copy
import json
import math
import os
import random
import sys
import time
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import pandas as pd


def find_workspace_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd() if start is None else start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "HighwayEnv").exists() and (candidate / "rl-agents").exists():
            return candidate
    raise FileNotFoundError("Could not find workspace root containing HighwayEnv and rl-agents")


ROOT = find_workspace_root()


def configure_local_imports() -> None:
    for path in [ROOT / "HighwayEnv", ROOT / "rl-agents"]:
        path_str = str(path)
        if path_str not in sys.path:
            sys.path.insert(0, path_str)


configure_local_imports()

import gymnasium as gym
from gymnasium import spaces
import highway_env  # noqa: F401, registers HighwayEnv gymnasium environments
from highway_env import utils
from highway_env.envs.common import abstract as abstract_module
from highway_env.envs.common import action as action_module
from highway_env.envs.common.action import ActionType
from highway_env.vehicle.controller import ControlledVehicle
from highway_env.vehicle.kinematics import Vehicle
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecMonitor

print(f"Workspace: {ROOT}")
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Environment

### 2.1 Action Type

#### 2.1.1 Acceleration-Only PPO

PPO controls acceleration. HighwayEnv handles steering with the deterministic route-following controller.

In [ ]:
_ORIGINAL_ACTION_FACTORY = getattr(
    action_module,
    "_codex_original_action_factory",
    action_module.action_factory,
)
action_module._codex_original_action_factory = _ORIGINAL_ACTION_FACTORY


class RouteFollowingAccelerationVehicle(ControlledVehicle):
    """Route-following steering with externally commanded acceleration."""

    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.target_acceleration = 0.0

    def act(self, action=None) -> None:
        if isinstance(action, dict) and "acceleration" in action:
            self.target_acceleration = float(action["acceleration"])

        self.follow_road()
        steering = self.steering_control(self.target_lane_index)
        steering = float(np.clip(steering, -self.MAX_STEERING_ANGLE, self.MAX_STEERING_ANGLE))
        Vehicle.act(self, {"steering": steering, "acceleration": self.target_acceleration})


class LongitudinalContinuousAction(ActionType):
    """One-dimensional continuous acceleration action for PPO."""

    ACCELERATION_RANGE = (-5.0, 5.0)

    def __init__(self, env, acceleration_range=None, speed_range=None, clip=True, **kwargs) -> None:
        super().__init__(env)
        self.acceleration_range = acceleration_range or self.ACCELERATION_RANGE
        self.speed_range = speed_range
        self.clip = clip
        self.last_action = np.zeros(1, dtype=np.float32)

    def space(self) -> spaces.Box:
        return spaces.Box(-1.0, 1.0, shape=(1,), dtype=np.float32)

    @property
    def vehicle_class(self):
        return RouteFollowingAccelerationVehicle

    def get_action(self, action: np.ndarray) -> dict:
        action = np.asarray(action, dtype=np.float32).reshape(-1)
        if self.clip:
            action = np.clip(action, -1.0, 1.0)
        acceleration = utils.lmap(float(action[0]), [-1.0, 1.0], self.acceleration_range)
        return {"acceleration": acceleration}

    def act(self, action: np.ndarray) -> None:
        command = self.get_action(action)
        if self.speed_range:
            self.controlled_vehicle.MIN_SPEED, self.controlled_vehicle.MAX_SPEED = self.speed_range
        self.controlled_vehicle.act(command)
        self.last_action = np.asarray(action, dtype=np.float32).reshape(1)

    def get_available_actions(self):
        return []


def longitudinal_action_factory(env, config: dict) -> ActionType:
    if config.get("type") == "LongitudinalContinuousAction":
        return LongitudinalContinuousAction(env, **config)
    return _ORIGINAL_ACTION_FACTORY(env, config)


def register_longitudinal_action_type() -> None:
    action_module.action_factory = longitudinal_action_factory
    abstract_module.action_factory = longitudinal_action_factory


register_longitudinal_action_type()
print("Registered LongitudinalContinuousAction: PPO acceleration, deterministic route-following steering.")

### 2.2 Configuration

#### 2.2.1 Tuning Cell

Edit route, traffic, reward, duration, observation features, and acceleration bounds here.

In [ ]:
ENV_ID = "intersection-v1"
OBS_FEATURES = ["presence", "x", "y", "vx", "vy", "cos_h", "sin_h"]

# Route target in HighwayEnv intersection:
#   o1: left-turn-style destination from ego's default approach
#   o2: straight-style destination
#   o3: right-turn-style destination
DESTINATION = "o1"

# Continuous action normalization:
#   action[0] in [-1, 1] -> acceleration_range [m/s^2]
# Steering is not learned. It is computed by the deterministic route-following controller.
LONGITUDINAL_ACTION_CONFIG = {
    "type": "LongitudinalContinuousAction",
    "clip": True,
    "acceleration_range": [-5.0, 5.0],
    "speed_range": [0.0, 9.0],
}

ENV_CONFIG = {
    "observation": {
        "type": "Kinematics",
        "vehicles_count": 15,
        "features": OBS_FEATURES,
        "features_range": {
            "x": [-100, 100],
            "y": [-100, 100],
            "vx": [-20, 20],
            "vy": [-20, 20],
        },
        "absolute": True,
        "normalize": True,
        "flatten": False,
        "observe_intentions": False,
    },
    "action": LONGITUDINAL_ACTION_CONFIG,
    "policy_frequency": 5,
    "simulation_frequency": 15,
    "duration": 13,
    "destination": DESTINATION,
    "controlled_vehicles": 1,
    "initial_vehicle_count": 10,
    "spawn_probability": 0.6,
    "collision_reward": -5,
    "high_speed_reward": 1,
    "arrived_reward": 1,
    "reward_speed_range": [7.0, 9.0],
    "normalize_reward": False,
    "offroad_terminal": False,
    "screen_width": 600,
    "screen_height": 600,
    "centering_position": [0.5, 0.6],
    "scaling": 7.15,
    "render_agent": True,
    "offscreen_rendering": False,
}

SEED = 7
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

ENV_CONFIG

### 2.3 Direct HighwayEnv Creation

#### 2.3.1 Single-Environment Factory

In [ ]:
def make_intersection_env(config: Optional[dict] = None, seed: Optional[int] = None, render_mode: Optional[str] = None):
    configure_local_imports()
    import gymnasium as gym
    import highway_env  # noqa: F401
    register_longitudinal_action_type()

    env_kwargs = {"render_mode": render_mode}
    if config is not None:
        env_kwargs["config"] = copy.deepcopy(config)
    env = gym.make(ENV_ID, **env_kwargs)
    obs, info = env.reset(seed=seed)
    return env, obs, info


env, obs, info = make_intersection_env(ENV_CONFIG, seed=SEED)
print("observation shape:", obs.shape)
print("observation space:", env.observation_space)
print("action space:", env.action_space)
print("sample normalized acceleration action:", env.action_space.sample())
env.close()

## 3. RL Problem

### 3.1 MDP

#### 3.1.1 State

`s_t`: `(15, 7)` kinematics tensor. Features:

`presence, x, y, vx, vy, cos_h, sin_h`

SB3 `MlpPolicy` flattens this to `105` features.

#### 3.1.2 Action

`a_t`: scalar normalized acceleration in `[-1, 1]`, mapped to `[-5, 5] m/s^2`; speed is capped to `[0, 9] m/s`. Steering is deterministic route-following.

#### 3.1.3 Reward

Native HighwayEnv reward: collision `-5`, high-speed `+1`, arrival `+1`, no normalization.

#### 3.1.4 Termination

Terminate on collision or arrival. Truncate at `duration`.

### 3.2 PPO Objective

#### 3.2.1 Training Target

Maximize discounted return with PPO. UTD stays `1` using `n_epochs=1` and `batch_size=rollout_size`.

## 4. PPO Hyperparameters

### 4.1 Rollout Geometry

#### 4.1.1 Vectorization, GPU, UTD

In [ ]:
CPU_COUNT = os.cpu_count() or 1
TARGET_TOTAL_TIMESTEPS = 100_000
TARGET_PPO_UPDATES = 50
PREFERRED_N_ENVS = 8
N_ENVS = next(
    n
    for n in range(min(PREFERRED_N_ENVS, CPU_COUNT), 0, -1)
    if TARGET_TOTAL_TIMESTEPS % (n * TARGET_PPO_UPDATES) == 0
)
VEC_BACKEND = "subproc"  # use "dummy" if a Windows/Jupyter subprocess issue appears

USE_CUDA = True
DEVICE = "cuda" if USE_CUDA and torch.cuda.is_available() else "cpu"

N_STEPS_PER_ENV = TARGET_TOTAL_TIMESTEPS // (N_ENVS * TARGET_PPO_UPDATES)
ROLLOUT_SIZE = N_ENVS * N_STEPS_PER_ENV
TOTAL_TIMESTEPS = ROLLOUT_SIZE * TARGET_PPO_UPDATES
UTD_RATIO = 1

PPO_CONFIG = {
    "policy": "MlpPolicy",
    "n_steps": N_STEPS_PER_ENV,
    "batch_size": ROLLOUT_SIZE,
    "n_epochs": UTD_RATIO,
    "gamma": 0.985,
    "gae_lambda": 0.92,
    "learning_rate": 2.5e-4,
    "clip_range": 0.2,
    "ent_coef": 0.015,
    "vf_coef": 0.5,
    "max_grad_norm": 0.5,
    "device": DEVICE,
    "verbose": 1,
}

POLICY_KWARGS = {
    "net_arch": {
        "pi": [256, 256],
        "vf": [256, 256],
    }
}

assert TOTAL_TIMESTEPS == TARGET_TOTAL_TIMESTEPS
assert N_STEPS_PER_ENV > 0
assert PPO_CONFIG["n_epochs"] == 1, "UTD ratio must remain 1: keep PPO n_epochs=1."
assert PPO_CONFIG["batch_size"] == ROLLOUT_SIZE, "Use the full rollout batch to keep one optimizer pass per rollout."

print(f"N_ENVS={N_ENVS}, backend={VEC_BACKEND}, n_steps={N_STEPS_PER_ENV}, rollout_size={ROLLOUT_SIZE}")
print(f"total_timesteps={TOTAL_TIMESTEPS}, PPO updates={TARGET_PPO_UPDATES}")
print(f"device={DEVICE}, UTD ratio={PPO_CONFIG['n_epochs']}")

## 5. Metrics

### 5.1 Core Metrics

#### 5.1.1 Episode-Level

### 5.2 Intersection Metrics

#### 5.2.1 Safety, Progress, Control

- `success_rate`: arrived without collision.
- `collision_rate`: ego crashed.
- `timeout_rate`: truncated without arrival or crash.
- `min_ttc`: approximate minimum TTC from kinematics features.
- `mean_ego_speed`, `high_speed_rate`.
- `terminal_speed`, `terminal_reward`.
- `mean_abs_accel_action`, `brake_action_rate`, `throttle_action_rate`.

In [ ]:
FEATURE_INDEX = {name: index for index, name in enumerate(OBS_FEATURES)}


def ego_speed_from_env(env) -> float:
    vehicle = getattr(env.unwrapped, "vehicle", None)
    return float(getattr(vehicle, "speed", 0.0))


def min_ttc_from_env(env, max_distance: float = 70.0) -> float:
    ego = getattr(env.unwrapped, "vehicle", None)
    road = getattr(env.unwrapped, "road", None)
    if ego is None or road is None:
        return math.inf

    ego_pos = np.asarray(ego.position, dtype=np.float32)
    ego_vel = np.asarray(ego.velocity, dtype=np.float32)
    best = math.inf
    for other in road.vehicles:
        if other is ego:
            continue
        rel_pos = np.asarray(other.position, dtype=np.float32) - ego_pos
        distance = float(np.linalg.norm(rel_pos))
        if distance < 1e-3 or distance > max_distance:
            continue
        rel_vel = np.asarray(other.velocity, dtype=np.float32) - ego_vel
        closing_speed = -float(np.dot(rel_pos, rel_vel)) / distance
        if closing_speed > 1e-3:
            best = min(best, distance / closing_speed)
    return best


def has_arrived(env, info: Optional[dict] = None) -> bool:
    rewards = (info or {}).get("rewards", {}) if isinstance(info, dict) else {}
    if isinstance(rewards, dict) and bool(rewards.get("arrived_reward", False)):
        return True
    vehicle = getattr(env.unwrapped, "vehicle", None)
    has_arrived_fn = getattr(env.unwrapped, "has_arrived", None)
    if vehicle is not None and callable(has_arrived_fn):
        try:
            return bool(has_arrived_fn(vehicle))
        except Exception:
            return False
    return False


def summarize_episodes(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    return pd.DataFrame(
        [
            {
                "episodes": len(df),
                "success_rate": df["success"].mean(),
                "collision_rate": df["collision"].mean(),
                "timeout_rate": df["timeout"].mean(),
                "mean_return": df["return"].mean(),
                "mean_discounted_return": df["discounted_return"].mean(),
                "mean_steps": df["steps"].mean(),
                "mean_survival_time_s": df["survival_time_s"].mean(),
                "mean_min_ttc": df["min_ttc"].replace(np.inf, np.nan).mean(),
                "mean_ego_speed": df["mean_ego_speed"].mean(),
                "mean_terminal_speed": df["terminal_speed"].mean(),
                "mean_terminal_reward": df["terminal_reward"].mean(),
                "high_speed_rate": df["high_speed_rate"].mean(),
                "mean_abs_accel_action": df["mean_abs_accel_action"].mean(),
                "brake_action_rate": df["brake_action_rate"].mean(),
                "throttle_action_rate": df["throttle_action_rate"].mean(),
            }
        ]
    )

## 6. PPO Baseline Implementation

### 6.1 Vectorized Environment Factory

#### 6.1.1 Parallel Rollout Collection

In [ ]:
def make_env_thunk(rank: int, config: dict, base_seed: int):
    def _init():
        env, _, _ = make_intersection_env(config=config, seed=base_seed + rank)
        return env

    return _init


def make_vector_env(config: dict, n_envs: int, seed: int, backend: str = "subproc"):
    env_fns = [make_env_thunk(rank, config, seed) for rank in range(n_envs)]
    if backend == "subproc" and n_envs > 1:
        vec_env = SubprocVecEnv(env_fns, start_method="spawn")
    else:
        vec_env = DummyVecEnv(env_fns)
    return VecMonitor(vec_env)


vec_env = make_vector_env(ENV_CONFIG, n_envs=N_ENVS, seed=SEED, backend=VEC_BACKEND)
print("vectorized observation space:", vec_env.observation_space)
print("vectorized action space:", vec_env.action_space)
vec_env.close()

### 6.2 Single-Episode Evaluation

#### 6.2.1 Deterministic PPO Rollouts

In [ ]:
def run_eval_episode(
    model: Optional[PPO],
    env_config: dict,
    episode: int,
    seed: int,
    gamma: float,
    deterministic: bool = True,
) -> dict:
    env, obs, _ = make_intersection_env(env_config, seed=seed)
    done = False
    total_reward = 0.0
    discounted_return = 0.0
    steps = 0
    crashed = False
    arrived = False
    min_ttc = math.inf
    speed_sum = 0.0
    high_speed_steps = 0
    abs_accel_sum = 0.0
    brake_steps = 0
    throttle_steps = 0
    high_speed_threshold = float(env_config["reward_speed_range"][0])
    terminal_reward = 0.0
    terminal_speed = 0.0

    while not done:
        if model is None:
            action = env.action_space.sample()
        else:
            action, _ = model.predict(obs, deterministic=deterministic)
        action_arr = np.asarray(action, dtype=np.float32).reshape(-1)

        next_obs, reward, terminated, truncated, info = env.step(action)
        terminal = bool(terminated or truncated)

        speed = ego_speed_from_env(env)
        min_ttc = min(min_ttc, min_ttc_from_env(env))
        speed_sum += speed
        high_speed_steps += int(speed >= high_speed_threshold)
        if action_arr.size >= 1:
            abs_accel_sum += abs(float(action_arr[0]))
            brake_steps += int(float(action_arr[0]) < -0.25)
            throttle_steps += int(float(action_arr[0]) > 0.25)
        total_reward += float(reward)
        discounted_return += (gamma ** steps) * float(reward)
        if terminal:
            terminal_reward = float(reward)
            terminal_speed = speed
        steps += 1
        crashed = crashed or bool(info.get("crashed", False))
        arrived = arrived or has_arrived(env, info)
        obs = next_obs
        done = terminal

    env.close()
    return {
        "episode": episode,
        "return": total_reward,
        "discounted_return": discounted_return,
        "steps": steps,
        "survival_time_s": steps / float(env_config["policy_frequency"]),
        "success": bool(arrived and not crashed),
        "collision": bool(crashed),
        "timeout": bool((not arrived) and (not crashed)),
        "min_ttc": min_ttc,
        "mean_ego_speed": speed_sum / max(steps, 1),
        "terminal_speed": terminal_speed,
        "terminal_reward": terminal_reward,
        "high_speed_rate": high_speed_steps / max(steps, 1),
        "mean_abs_accel_action": abs_accel_sum / max(steps, 1),
        "brake_action_rate": brake_steps / max(steps, 1),
        "throttle_action_rate": throttle_steps / max(steps, 1),
    }


def evaluate_policy(
    model: Optional[PPO],
    env_config: dict,
    episodes: int,
    seed: int,
    gamma: float,
    deterministic: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rows = [
        run_eval_episode(model, env_config, episode, seed + episode, gamma, deterministic=deterministic)
        for episode in range(episodes)
    ]
    df = pd.DataFrame(rows)
    return df, summarize_episodes(df)

### 6.3 Train PPO

#### 6.3.1 GPU-Enabled Vectorized Training

In [ ]:
def train_ppo_baseline(
    env_config: dict,
    total_timesteps: int,
    seed: int,
    n_envs: int,
    backend: str,
) -> Tuple[PPO, pd.DataFrame]:
    vec_env = make_vector_env(env_config, n_envs=n_envs, seed=seed, backend=backend)
    model = PPO(
        PPO_CONFIG["policy"],
        vec_env,
        n_steps=PPO_CONFIG["n_steps"],
        batch_size=PPO_CONFIG["batch_size"],
        n_epochs=PPO_CONFIG["n_epochs"],
        gamma=PPO_CONFIG["gamma"],
        gae_lambda=PPO_CONFIG["gae_lambda"],
        learning_rate=PPO_CONFIG["learning_rate"],
        clip_range=PPO_CONFIG["clip_range"],
        ent_coef=PPO_CONFIG["ent_coef"],
        vf_coef=PPO_CONFIG["vf_coef"],
        max_grad_norm=PPO_CONFIG["max_grad_norm"],
        policy_kwargs=POLICY_KWARGS,
        seed=seed,
        device=PPO_CONFIG["device"],
        verbose=PPO_CONFIG["verbose"],
    )

    start = time.time()
    model.learn(total_timesteps=total_timesteps, progress_bar=False)
    wall_time_s = time.time() - start
    vec_env.close()

    train_info = pd.DataFrame(
        [
            {
                "total_timesteps": total_timesteps,
                "n_envs": n_envs,
                "backend": backend,
                "device": PPO_CONFIG["device"],
                "n_steps_per_env": PPO_CONFIG["n_steps"],
                "rollout_size": ROLLOUT_SIZE,
                "ppo_updates": TARGET_PPO_UPDATES,
                "batch_size": PPO_CONFIG["batch_size"],
                "n_epochs": PPO_CONFIG["n_epochs"],
                "utd_ratio": PPO_CONFIG["n_epochs"],
                "wall_time_s": wall_time_s,
            }
        ]
    )
    return model, train_info

## 7. Run PPO Baseline

### 7.1 Training Run

#### 7.1.1 100k Training Run

In [ ]:
EVAL_EPISODES = 20

ppo_model, train_info = train_ppo_baseline(
    env_config=ENV_CONFIG,
    total_timesteps=TOTAL_TIMESTEPS,
    seed=SEED,
    n_envs=N_ENVS,
    backend=VEC_BACKEND,
)
display(train_info)

### 7.2 Evaluation Run

#### 7.2.1 Deterministic Evaluation

In [ ]:
ppo_eval_df, ppo_summary = evaluate_policy(
    model=ppo_model,
    env_config=ENV_CONFIG,
    episodes=EVAL_EPISODES,
    seed=SEED + 20_000,
    gamma=float(PPO_CONFIG["gamma"]),
    deterministic=True,
)
ppo_summary.insert(0, "agent", "ppo_continuous_eval")
display(ppo_summary)

summary = ppo_summary.copy()
summary

## 8. Render Learned Policy

### 8.1 Policy Rollout Video

#### 8.1.1 Deterministic PPO Render

In [ ]:
def collect_policy_frames(
    model: PPO,
    env_config: dict,
    seed: int,
    deterministic: bool = True,
) -> list[np.ndarray]:
    render_config = copy.deepcopy(env_config)
    render_config.update(
        {
            "offscreen_rendering": True,
            "show_trajectories": True,
            "screen_width": 800,
            "screen_height": 800,
        }
    )
    env, obs, info = make_intersection_env(render_config, seed=seed, render_mode="rgb_array")
    max_steps = int(render_config["duration"] * render_config["policy_frequency"]) + 1
    frames = []

    try:
        for _ in range(max_steps):
            frame = env.render()
            if frame is not None:
                frames.append(frame)

            action, _ = model.predict(obs, deterministic=deterministic)
            obs, reward, terminated, truncated, info = env.step(action)

            if terminated or truncated:
                frame = env.render()
                if frame is not None:
                    frames.append(frame)
                break
    finally:
        env.close()

    if not frames:
        raise RuntimeError("No frames were rendered. Check pygame/offscreen rendering setup.")
    return frames


RENDER_DIR = ROOT / "notebooks" / "results" / "Intersection"
RENDER_DIR.mkdir(parents=True, exist_ok=True)
LEARNED_POLICY_GIF = RENDER_DIR / "ppo_learned_policy.gif"

learned_policy_frames = collect_policy_frames(
    ppo_model,
    ENV_CONFIG,
    seed=SEED + 30_000,
    deterministic=True,
)

try:
    import imageio.v2 as imageio
    from IPython.display import Image, display

    imageio.mimsave(
        str(LEARNED_POLICY_GIF),
        learned_policy_frames,
        duration=1.0 / float(ENV_CONFIG["policy_frequency"]),
    )
    display(Image(filename=str(LEARNED_POLICY_GIF)))
    print(f"Saved learned-policy render to {LEARNED_POLICY_GIF}")
except Exception as exc:
    import matplotlib.pyplot as plt
    from IPython.display import HTML, display
    from matplotlib import animation

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.axis("off")
    image = ax.imshow(learned_policy_frames[0])

    def update_frame(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update_frame,
        frames=learned_policy_frames,
        interval=1000.0 / float(ENV_CONFIG["policy_frequency"]),
        blit=True,
    )
    display(HTML(anim.to_jshtml()))
    plt.close(fig)
    print(f"GIF export skipped: {exc}")

## 9. Save Artifacts

### 9.1 Compact Results

#### 9.1.1 CSVs, Configs, And Model

In [ ]:
RESULTS_DIR = ROOT / "notebooks" / "results" / "Intersection"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ppo_eval_df.to_csv(RESULTS_DIR / "ppo_continuous_eval_episodes.csv", index=False)
summary.to_csv(RESULTS_DIR / "summary.csv", index=False)
train_info.to_csv(RESULTS_DIR / "ppo_continuous_train_info.csv", index=False)

with (RESULTS_DIR / "env_config.json").open("w", encoding="utf-8") as f:
    json.dump(ENV_CONFIG, f, indent=2)
with (RESULTS_DIR / "ppo_config.json").open("w", encoding="utf-8") as f:
    json.dump({**PPO_CONFIG, "policy_kwargs": POLICY_KWARGS}, f, indent=2, default=str)

model_path = RESULTS_DIR / "ppo_continuous_intersection.zip"
ppo_model.save(model_path)

print(f"Saved compact results to {RESULTS_DIR}")
print(f"Saved model to {model_path}")

## 10. Results Log

### 10.1 Configuration

`intersection-v1`, accel-only PPO, speed cap `[0, 9]`, deterministic steering, vectorized envs, CUDA if available, UTD `1`.

### 10.2 Result Table

| date | config tag | timesteps | n_envs | device | success | collision | timeout | mean return | notes |
| --- | --- | ---: | ---: | --- | ---: | ---: | ---: | ---: | --- |
| | | | | | | | | | |